In [90]:
import numpy as np
import pandas as pd

In [91]:
df=pd.read_csv("retail_sales_dirty_dataset.csv")

In [92]:
df.head(55)

,Order_ID,Order_Date,Customer_Name,City,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Sales,Profit,Payment_Method
0,1001,NaN,Anjali Mehta,Hyderabad,North,Electronics,Laptop,five,7217,4467.0,78563,NaN,Cash
1,1002,2024-02-05,Simran Kaur,Jaipur,North,Home Appliances,AC,8,925,1307.0,52052,6074.0,Debit Card
2,1003,2024-03-20,Riya Malhotra,Jaipur,North,Clothing,T-Shirt,8,25307,NaN,11006,10799.0,Cash
3,1004,2024-01-27,Rahul Khan,Mumbai,West,Clothing,T-Shirt,4,30214,2988.0,26417,6320.0,Credit Card
4,1005,2024-02-10,Anjali Mehta,Mumbai,South,Home Appliances,Washing Machine,nine,50852,1876.0,32869,13689.0,Credit Card
5,1006,2024-02-21,Priya Singh,Bangalore,North,Home Appliances,Refrigerator,three,37289,4788.0,117866,10060.0,UPI
6,1007,2024-04-20,Karan Patel,Mumbai,South,Furniture,Table,7,39552,4334.0,105741,9564.0,Debit Card
7,1008,2024-01-23,Neha Verma,Mumbai,West,Electronics,Keyboard,seven,not_available,2157.0,25032,12983.0,Debit Card
8,1009,2024-04-16,Anjali Mehta,NaN,North,Furniture,Bookshelf,9,32521,NaN,16226,14896.0,Credit Card
9,1010,2024-01-28,Simran Kaur,Hyderabad,South,Electronics,Headphones,three,36531,4322.0,62422,7432.0,Credit Card


In [107]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          100 non-null    int64  
 1   Order_Date        100 non-null    object 
 2   Customer_Name     100 non-null    object 
 3   City              100 non-null    object 
 4   Region            100 non-null    object 
 5   Product_Category  100 non-null    object 
 6   Product_Name      100 non-null    object 
 7   Quantity          100 non-null    int64  
 8   Unit_Price        100 non-null    float64
 9   Discount          100 non-null    float64
 10  Sales             100 non-null    int64  
 11  Profit            100 non-null    float64
 12  Payment_Method    100 non-null    object 
dtypes: float64(3), int64(3), object(7)
memory usage: 10.3+ KB


In [94]:
df.isnull().sum()

Order_ID             0
Order_Date           1
Customer_Name        0
City                11
Region               0
Product_Category     0
Product_Name         0
Quantity             0
Unit_Price           0
Discount            12
Sales                0
Profit              11
Payment_Method       0
dtype: int64

In [95]:
df["Order_Date"]=pd.to_datetime(df["Order_Date"],errors="coerce")  # converting Order_Date column into datetime datatype

In [96]:
df["Order_Date"]=df["Order_Date"].ffill()   # filling missing dates using previous valid date

In [97]:
from datetime import date #first we need to  import this 
df["Order_Date"] = df["Order_Date"].fillna(date.today()) #there is a problem that 1st row also has missng value so filling first missing date with today's date

In [98]:
df["City"]=df["City"].fillna("Not Available")  # filling missing city values with Not Available

In [99]:
from word2number import w2n  # converting text numbers like five and seven into numeric values

df["Quantity"] = df["Quantity"].apply(
    lambda x: w2n.word_to_num(x)
    if isinstance(x, str)
    else x
)

In [100]:
df["Quantity"] = pd.to_numeric(df["Quantity"]) # converting Quantity column datatype into numeric

In [101]:
df["Unit_Price"]=df["Unit_Price"].replace("not_available",pd.NA)  # replacing not_available values with NaN

In [102]:
df["Unit_Price"]=pd.to_numeric(df["Unit_Price"],errors="coerce")  # converting Unit_Price datatype into numeric

In [103]:
df["Unit_Price"] = df.groupby(          # filling missing Unit_Price values using Product_Name wise mean
    "Product_Name"
)["Unit_Price"].transform(
    lambda x: x.fillna(x.mean())
)

In [108]:
df["Discount"] = df.groupby(              # filling missing Discount values using Product_Category wise mean
    "Product_Category"
)["Discount"].transform(
    lambda x: x.fillna(x.mean())
)

In [109]:
df["Profit"]=df.groupby("Product_Category")["Profit"].transform(lambda x: x.fillna(x.mean()))   # filling missing Profit values using Product_Category wise mean

In [123]:
#automation
def clean_data(df):

     # converting Order_Date column into datetime datatype
    df["Order_Date"] = pd.to_datetime(
        df["Order_Date"],
        errors="coerce"
    )

    # filling missing dates using previous valid date
    df["Order_Date"] = df["Order_Date"].ffill()

    # filling first missing date with today's date
    df["Order_Date"] = df["Order_Date"].fillna(date.today())



    # filling missing city values with Not Available
    df["City"] = df["City"].fillna("Not Available")



    # converting text numbers like five and seven into numeric values
    df["Quantity"] = df["Quantity"].apply(

        lambda x: w2n.word_to_num(x)

        if isinstance(x, str)

        else x
    )

    # converting Quantity column datatype into numeric
    df["Quantity"] = pd.to_numeric(
        df["Quantity"],
        errors="coerce"
    )



    # replacing not_available values with NaN
    df["Unit_Price"] = df["Unit_Price"].replace(
        "not_available",
        np.nan
    )

    # converting Unit_Price datatype into numeric
    df["Unit_Price"] = pd.to_numeric(
        df["Unit_Price"],
        errors="coerce"
    )

    # filling missing Unit_Price values using Product_Name wise mean
    df["Unit_Price"] = df.groupby(
        "Product_Name"
    )["Unit_Price"].transform(

        lambda x: x.fillna(x.mean())
    )



    # converting Discount datatype into numeric
    df["Discount"] = pd.to_numeric(
        df["Discount"],
        errors="coerce"
    )

    # filling missing Discount values using Product_Category wise mean
    df["Discount"] = df.groupby(
        "Product_Category"
    )["Discount"].transform(

        lambda x: x.fillna(x.mean())
    )



    # converting Profit datatype into numeric
    df["Profit"] = pd.to_numeric(
        df["Profit"],
        errors="coerce"
    )

    # filling missing Profit values using Product_Category wise mean
    df["Profit"] = df.groupby(
        "Product_Category"
    )["Profit"].transform(

        lambda x: x.fillna(x.mean())
    )


    return df

In [124]:
cleaned_df = clean_data(df)

In [125]:
cleaned_df.head()

,Order_ID,Order_Date,Customer_Name,City,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Sales,Profit,Payment_Method
0,1001,2026-05-09,Anjali Mehta,Hyderabad,North,Electronics,Laptop,5,7217.0,4467.000000,78563,8162.090909,Cash
1,1002,2024-02-05,Simran Kaur,Jaipur,North,Home Appliances,AC,8,925.0,1307.000000,52052,6074.000000,Debit Card
2,1003,2024-03-20,Riya Malhotra,Jaipur,North,Clothing,T-Shirt,8,25307.0,2287.166667,11006,10799.000000,Cash
3,1004,2024-01-27,Rahul Khan,Mumbai,West,Clothing,T-Shirt,4,30214.0,2988.000000,26417,6320.000000,Credit Card
4,1005,2024-02-10,Anjali Mehta,Mumbai,South,Home Appliances,Washing Machine,9,50852.0,1876.000000,32869,13689.000000,Credit Card


In [126]:
cleaned_df.isnull().sum()

Order_ID            0
Order_Date          0
Customer_Name       0
City                0
Region              0
Product_Category    0
Product_Name        0
Quantity            0
Unit_Price          0
Discount            0
Sales               0
Profit              0
Payment_Method      0
dtype: int64

In [127]:
#Load Test Dataset
test_df = pd.read_csv(
    "test_dirty_retail_data.csv"
)

In [128]:
test_df.head(20)

,Order_ID,Order_Date,Customer_Name,City,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Sales,Profit,Payment_Method
0,2001,2024-05-01,Aryan Mehta,Delhi,North,Electronics,Laptop,five,55000,5000.0,105000,15000.0,Credit Card
1,2002,NaN,Priya Sharma,NaN,West,Furniture,Chair,three,not_available,NaN,17000,NaN,UPI
2,2003,invalid_date,Rahul Khan,Mumbai,West,Electronics,Keyboard,seven,1500,200.0,22000,5000.0,Debit Card
3,2004,2024-05-04,Neha Verma,Jaipur,North,Clothing,Jacket,4,NaN,500.0,18000,3000.0,Cash
4,2005,2024-05-05,NaN,Hyderabad,South,Home Appliances,Microwave,two,12000,NaN,25000,NaN,Credit Card
5,2006,NaN,Simran Kaur,Bangalore,South,Electronics,Mouse,eight,800,NaN,15000,4000.0,NaN
6,2007,2024-05-07,Karan Patel,NaN,East,Furniture,Table,6,25000,1000.0,45000,8000.0,UPI
7,2008,invalid_date,Arjun Mehta,Pune,West,Electronics,Headphones,nine,not_available,300.0,27000,NaN,Debit Card
8,2009,2024-05-09,Riya Malhotra,Chennai,South,Clothing,T-Shirt,one,2000,NaN,12000,2500.0,Cash
9,2010,NaN,Aman Sharma,Kolkata,East,Home Appliances,Refrigerator,ten,42000,4000.0,85000,16000.0,Credit Card


In [129]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          15 non-null     int64  
 1   Order_Date        10 non-null     object 
 2   Customer_Name     14 non-null     object 
 3   City              12 non-null     object 
 4   Region            15 non-null     object 
 5   Product_Category  15 non-null     object 
 6   Product_Name      15 non-null     object 
 7   Quantity          15 non-null     object 
 8   Unit_Price        13 non-null     object 
 9   Discount          8 non-null      float64
 10  Sales             15 non-null     int64  
 11  Profit            10 non-null     float64
 12  Payment_Method    14 non-null     object 
dtypes: float64(2), int64(2), object(9)
memory usage: 1.7+ KB


In [130]:
test_df.isnull().sum()

Order_ID            0
Order_Date          5
Customer_Name       1
City                3
Region              0
Product_Category    0
Product_Name        0
Quantity            0
Unit_Price          2
Discount            7
Sales               0
Profit              5
Payment_Method      1
dtype: int64

In [131]:
cleaned_test_df = clean_data(test_df) #Run Automation

In [132]:
cleaned_test_df.isnull().sum()

Order_ID            0
Order_Date          0
Customer_Name       1
City                0
Region              0
Product_Category    0
Product_Name        0
Quantity            0
Unit_Price          2
Discount            0
Sales               0
Profit              0
Payment_Method      1
dtype: int64

In [122]:
cleaned_test_df.head(20)

,Order_ID,Order_Date,Customer_Name,City,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Sales,Profit,Payment_Method
0,2001,2024-05-01,Aryan Mehta,Delhi,North,Electronics,Laptop,5,55000.0,5000.0,105000,15000.0,Credit Card
1,2002,2024-05-01,Priya Sharma,Not Available,West,Furniture,Chair,3,7000.0,750.0,17000,8000.0,UPI
2,2003,2024-05-01,Rahul Khan,Mumbai,West,Electronics,Keyboard,7,1500.0,200.0,22000,5000.0,Debit Card
3,2004,2024-05-04,Neha Verma,Jaipur,North,Clothing,Jacket,4,3500.0,500.0,18000,3000.0,Cash
4,2005,2024-05-05,NaN,Hyderabad,South,Home Appliances,Microwave,2,12000.0,4000.0,25000,11000.0,Credit Card
5,2006,2024-05-05,Simran Kaur,Bangalore,South,Electronics,Mouse,8,800.0,1425.0,15000,4000.0,NaN
6,2007,2024-05-07,Karan Patel,Not Available,East,Furniture,Table,6,25000.0,1000.0,45000,8000.0,UPI
7,2008,2024-05-07,Arjun Mehta,Pune,West,Electronics,Headphones,9,NaN,300.0,27000,7050.0,Debit Card
8,2009,2024-05-09,Riya Malhotra,Chennai,South,Clothing,T-Shirt,1,2000.0,500.0,12000,2500.0,Cash
9,2010,2024-05-09,Aman Sharma,Kolkata,East,Home Appliances,Refrigerator,10,42000.0,4000.0,85000,16000.0,Credit Card


In [133]:
cleaned_test_df.to_csv(
    "cleaned_test_data.csv",
    index=False
)